        # 🔤 L04　字串與格式化輸出入
        **Python 冒險之旅 2026**　｜　Day 2（08/30 日）🌴 文字之島　｜　關卡　｜　🏅 100 XP

        📖 對應教科書：第 3 章 3.1–3.4


        ### 🎯 這一關你會學到
        - 字串索引、切片與常用方法
- 用 input() 取得使用者輸入並轉型
- 用 format() 與 f-string 對齊、補零、千分位

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/python-quest-2026/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  Python 冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins

_LEVEL = "L04"
_SALT = "python-quest-2026-datama"
_TASKS = ["4-1", "4-2", "4-3", "4-4", "4-5", "4-6"]
_XP_EACH = 16
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_pyquest_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

def 行列表(out):
    return [ln.rstrip() for ln in str(out).splitlines() if ln.strip()]

class _NeedMoreInput(Exception):
    pass

_HIST = builtins.__dict__.setdefault("_pyquest_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_pyquest_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_pyquest_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

def _find_cell(tid):
    marker = "# 🎯 任務 " + tid
    for cell in reversed(_history()):
        if marker in cell:
            lines = [ln for ln in cell.splitlines()
                     if not re.match(r"\s*(檢查|通關密語)\s*\(", ln)]
            return "\n".join(lines)
    return None

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                _plt.show = _orig_show
        return buf.getvalue(), ns
    run.src = src
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _progress():
    done = sum(1 for t in _TASKS if _PASSED.get(t))
    bar = "■" * done + "□" * (len(_TASKS) - done)
    return f"[{bar}] {done}/{len(_TASKS)}"

def 檢查(tid):
    tid = str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    src = _find_cell(tid)
    if src is None:
        print(f"❌ 找不到「# 🎯 任務 {tid}」的程式格。請先執行那一格（並保留第一行的標記），再執行這裡。")
        return
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        result = (False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。")
    except Exception as e:
        tb = traceback.format_exc().strip().splitlines()[-1]
        result = (False, f"程式執行時發生錯誤 → {tb}")
    ok, extra = (result, "") if isinstance(result, bool) else result
    if ok:
        first = not _PASSED.get(tid)
        _PASSED[tid] = True
        print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")
        if all(_PASSED.get(t) for t in _TASKS):
            print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
    else:
        print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
        if extra: print("   💬 " + str(extra))
        if _HINTS.get(tid): print("   💡 提示：" + _HINTS[tid])
        print("   👉 修改程式後，先重新執行任務那一格，再執行這一格。")

def 通關密語():
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_SALT}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：PYQ-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_4_1(run):
    out, ns = run()
    lines = 行列表(out)
    need = ["P", "課", "Python", "課修必礎基 nohtyP"]
    missing = [n for n in need if n not in lines]
    return (not missing, f"還缺這些輸出：{missing}")
任務定義("4-1", _check_4_1, 提示="s[-1] 是最後一個字元；s[:6] 取前 6 個；s[::-1] 反轉。")

def _check_4_2(run):
    out, ns = run()
    lines = 行列表(out)
    need = ["HELLO WORLD", "Hello World", "hello python", "3", "4"]
    missing = [n for n in need if n not in lines]
    return (not missing, f"還缺這些輸出：{missing}")
任務定義("4-2", _check_4_2, 提示="s.title() → Hello World；s.count('l') → 3；s.find('o') → 4。")

def _check_4_3(run):
    out, ns = run("小明", "20")
    if not 出現(out, "小明"): return (False, "要印出輸入的姓名。")
    if not 出現(out, "21"): return (False, "明年的年齡要是 21（年齡要先用 int() 轉成整數）。")
    out2, ns2 = run("阿華", "35")
    return (出現(out2, "阿華", "36"), "換個輸入（阿華、35）也要正確。")
任務定義("4-3", _check_4_3, 提示="age = int(input('請輸入年齡：'))，明年就是 age + 1。")

def _check_4_4(run):
    out, ns = run()
    lines = 行列表(out)
    want = ["   水泥股    9.45    10,350", "   食品股   50.50     2,235", "   電子股  128.00   125,000"]
    missing = [w for w in want if w not in lines]
    return (not missing, f"這幾行的格式不對：{missing}（注意寬度、小數位數與千分位）。")
任務定義("4-4", _check_4_4, 提示="fmt.format(名稱, 價格, 成交量)，三行都用同一個 fmt。")

def _check_4_5(run):
    out, ns = run("25")
    if " 77.00" not in out: return (False, "輸入 25 時應該顯示  77.00（寬 6、小數 2 位）。")
    out2, ns2 = run("100")
    return ("212.00" in out2, "輸入 100 時應該顯示 212.00。")
任務定義("4-5", _check_4_5, 提示="f = c * 9 / 5 + 32；格式 {f:6.2f}。")

def _check_4_6(run):
    for word, ans in [("level", "True"), ("eye", "True"), ("Eye", "False"), ("noon", "True"), ("python", "False")]:
        out, ns = run(word)
        if ans not in 行列表(out)[-1:]:
            return (False, f"輸入 {word} 時應該印出 {ans}。")
    return True
任務定義("4-6", _check_4_6, 提示="s[::-1] 是反轉後的字串，s == s[::-1] 的結果就是 True/False。")


## 🔤 4-1　字串是「字元的序列」
字串裡每個字元都有**編號（索引）**，從 0 開始；從右邊數則是 -1、-2……
用 `[]` 取單一字元、用 `[開始:結束:間隔]` 取一段（**切片**，不包含結束位置）。
```
 P  y  t  h  o  n     基  礎  必  修  課
 0  1  2  3  4  5  6  7  8  9  10 11
-12 -11 ...                    -2 -1
```

In [ ]:
s = 'Python 基礎必修課'
print(s[0], s[3], s[-1], s[-2])     # P h 課 修
print(s[:6])      # 'Python'：從頭到索引 5
print(s[7:])      # '基礎必修課'：索引 7 到最後
print(s[::2])     # 每隔一個取：'Pto 礎修'
print(s[::-1])    # 反轉！
print(len(s))     # 字串長度 12
print('Py' + 'thon', '-' * 10)   # + 串接、* 重複

## 4-2　`input()`：跟使用者互動
`input('提示文字')` 會等使用者輸入，**永遠傳回字串**。要算數就得先 `int()` 或 `float()`。
在 Colab 執行時，程式格下方會出現輸入框，打完按 Enter。

In [ ]:
name = input('請輸入姓名：')
age = int(input('請輸入年齡：'))      # 課本 ex03/input.py
print('姓名：%s\t年齡：%d歲' % (name, age))
print(f'{name} 明年 {age + 1} 歲')

## 4-3　格式化輸出：`format()` 與 f-string（課本 3.3）
| 語法 | 意思 | 例子 → 結果 |
|---|---|---|
| `{:>8}` | 靠右、寬 8 | `f'{"股":>8}'` → `       股` |
| `{:<8}` / `{:^8}` | 靠左 / 置中 | |
| `{:7.2f}` | 寬 7、小數 2 位 | `9.45` → `   9.45` |
| `{:8,}` | 寬 8、千分位 | `10350` → `  10,350` |
| `{:03d}` | 補 0 到 3 位 | `7` → `007` |
| `{:.1%}` | 百分比 | `0.256` → `25.6%` |
| `{:$^6}` | 用 $ 填滿置中 | `1000` → `$1000$` |

In [ ]:
s, i, p = '字串', 1000, 0.666666
print('{:.2f}'.format(p))        # 0.67   （課本 ex03/format_3.py）
print('{:,}'.format(i * i))      # 1,000,000
print('{:.2%}'.format(p))        # 66.67%
print('|{:6}|{:>6}|{:<6}|'.format(s, s, i))
print('{:$^6}'.format(i))
data = "水泥股,9.45,10350"
ary = data.split(",")
print("{:>8}{:7.2f}{:8,}".format(ary[0], float(ary[1]), int(ary[2])))   # 課本 test03_4
print(f"{'食品股':>8}{50.5:7.2f}{2235:8,}")

## 4-4　常用字串方法（課本 3.4）
字串方法用「點」呼叫：`s.upper()`。**字串本身不會被改變**，方法會傳回新字串。

In [ ]:
s2 = 'hello world!'
print(s2.upper(), s2.title(), s2.capitalize())
print(s2.replace('world', 'python'))
print(s2.find('o'), s2.rfind('o'), s2.count('l'))   # 4 7 3
print(s2.startswith('hello'), s2.endswith('?'))
print('  多餘空白  '.strip() + '|')
print('人之初,性本善,性相近'.split(','))              # 切成串列
print('3M'.isalnum(), '3M'.isalpha(), '123'.isdigit())
print(len(s2), max(s2), min(s2))

### 🎯 任務 4-1　切片練習

`s = 'Python 基礎必修課'`。請依序印出：第一個字元、最後一個字元、前 6 個字元（`Python`）、整個字串反轉。

In [ ]:
# 🎯 任務 4-1　切片練習（請保留這一行）
s = 'Python 基礎必修課'
print(s[0])
# 繼續印出：最後一個字元、前 6 個字元、反轉的字串

In [ ]:
檢查("4-1")   # ◀ 執行這一格，看看任務 4-1 有沒有過關

### 🎯 任務 4-2　字串方法大集合

`s = 'hello world'`。請依序印出：全部大寫、每個單字字首大寫、把 world 換成 python、字母 `l` 出現的次數、`o` 第一次出現的索引。

In [ ]:
# 🎯 任務 4-2　字串方法大集合（請保留這一行）
s = 'hello world'
print(s.upper())
# 繼續：title()、replace()、count('l')、find('o')

In [ ]:
檢查("4-2")   # ◀ 執行這一格，看看任務 4-2 有沒有過關

### 🎯 任務 4-3　自我介紹機

用 `input()` 讀取**姓名**和**年齡**，印出一行：`姓名：小明　年齡：20 歲，明年 21 歲`（年齡要轉成整數才能 +1）。

In [ ]:
# 🎯 任務 4-3　自我介紹機（請保留這一行）
name = input('請輸入姓名：')
age = ???
print(f"姓名：{name}　年齡：{age} 歲，明年 {???} 歲")

In [ ]:
檢查("4-3")   # ◀ 執行這一格，看看任務 4-3 有沒有過關

### 🎯 任務 4-4　股市報表

三檔股票資料如下，請用 **同一個格式字串** 印出對齊的三行：名稱靠右寬 6、價格寬 8 小數 2 位、成交量寬 10 加千分位。

```
   水泥股    9.45    10,350
   食品股   50.50     2,235
   電子股  128.00   125,000
```

In [ ]:
# 🎯 任務 4-4　股市報表（請保留這一行）
fmt = "{:>6}{:8.2f}{:10,}"
print(fmt.format("水泥股", 9.45, 10350))
# 用同一個 fmt 印出食品股（50.5, 2235）與電子股（128, 125000）

In [ ]:
檢查("4-4")   # ◀ 執行這一格，看看任務 4-4 有沒有過關

### 🎯 任務 4-5　溫度轉換器（輸入版）

讀取攝氏溫度（可含小數），換算華氏並印出 `攝氏 25.0 度 = 華氏  77.00 度`（華氏整數 3 位、小數 2 位，總寬 6）。公式：F = C × 9 / 5 + 32。

In [ ]:
# 🎯 任務 4-5　溫度轉換器（輸入版）（請保留這一行）
c = float(input('請輸入攝氏溫度：'))
f = ???
print(f"攝氏 {c} 度 = 華氏 {f:???} 度")

In [ ]:
檢查("4-5")   # ◀ 執行這一格，看看任務 4-5 有沒有過關

### 🎯 任務 4-6　迴文偵測器

讀取一個英文字串，判斷是否為**迴文**（正著讀、倒著讀一樣，區分大小寫），直接印出 `True` 或 `False`。提示：比較字串和它的反轉。

In [ ]:
# 🎯 任務 4-6　迴文偵測器（請保留這一行）
s = input('請輸入字串：')
print(???)

In [ ]:
檢查("4-6")   # ◀ 執行這一格，看看任務 4-6 有沒有過關

## 💡 挑戰題（不計分）
`eval()` 可以把字串當成算式計算：`eval('3 + 4 * 2')` → 11。試著做一個「輸入算式就算給你看」的小計算機。
（小心：不要對來路不明的字串用 eval！）

---
## 🔑 通關密語

全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：🔀 L05 選擇結構 if / elif / else** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/python-quest-2026/blob/main/notebooks/L05_if_else.ipynb)

回到入口網頁：https://johnnychao.github.io/python-quest-2026/